# WAI INPAINT-XL LAB

**Hipotese:** corrigir SO a roupa de uma personagem ja transformada em chibi, sem retransformar a personagem inteira.

`CHIBI JA GERADO -> mascara da roupa -> Waifu-Inpaint-XL -> resto intacto`

Nao substitui o `wai_illustrious_sdxl_v170`, que segue gerando o chibi de entrada. O laboratorio anterior continua valido e intocado.

**Ordem:** validar modelo -> validar nodes -> validar mascara -> mostrar grafo -> executar UM baseline -> medir localidade.


In [ ]:
#@title 0. PAINEL DO INPAINT LAB — edite so esta celula { display-mode: "form" }
#@markdown # WAI INPAINT-XL LAB — correcao localizada de figurino
#@markdown
#@markdown **Hipotese desta fase:** em vez de transformar a personagem inteira
#@markdown de novo, mascarar SO a roupa e deixar o Waifu-Inpaint-XL redesenhar
#@markdown apenas ela, preservando rosto, cabelo, pele, pose e composicao.
#@markdown
#@markdown `CHIBI JA GERADO -> mascara da roupa -> inpaint -> resto intacto`
#@markdown
#@markdown Este lab **nao substitui** o `wai_illustrious_sdxl_v170`, que segue
#@markdown gerando o chibi de entrada.

#@markdown ---
#@markdown ### ENTRADAS
CHARACTER_ID = "waifu_001"  #@param {type:"string"}
#@markdown `SOURCE_IMAGE`: o melhor chibi ja gerado. NAO e a personagem real.
SOURCE_IMAGE = "wai_chibi_candidate.png"  #@param {type:"string"}
#@markdown `OUTFIT_MASK`: mascara SEMANTICA (branco = pode redesenhar).
#@markdown Nao e o crop `outfit.png` — sao coisas diferentes.
OUTFIT_MASK = "outfit_mask.png"  #@param {type:"string"}
#@markdown Referencias (nao usadas no TESTE 1):
OUTFIT_REFERENCE = "outfit.png"  #@param {type:"string"}
FULL_BODY_REFERENCE = "full_body.png"  #@param {type:"string"}
FACE_REFERENCE = "face.png"  #@param {type:"string"}

#@markdown ---
#@markdown ### MODO DE REFERENCIA
#@markdown TESTE 1 = `NONE` (inpaint puro). Estabelece o baseline: o que o
#@markdown proprio modelo de inpainting consegue fazer sozinho.
#@markdown As outras opcoes exigem IP-Adapter e ainda NAO estao implementadas —
#@markdown selecionar uma delas BLOQUEIA, em vez de rodar algo diferente do
#@markdown que o nome promete.
REFERENCE_MODE = "NONE"  #@param ["NONE", "OUTFIT_REFERENCE", "FULL_BODY_REFERENCE", "FULL_BODY_PLUS_OUTFIT"]

#@markdown ---
#@markdown ### MASCARA
#@markdown Dilatacao da a margem para reconstruir bordas; feather suaviza a
#@markdown transicao. Valores conservadores: mascara grande demais volta a
#@markdown redesenhar a personagem inteira.
MASK_DILATION = 8  #@param {type:"slider", min:0, max:64, step:1}
MASK_FEATHER = 6  #@param {type:"slider", min:0, max:64, step:1}
#@markdown Canal do arquivo de mascara. `red` serve para PNG preto e branco.
MASK_CHANNEL = "red"  #@param ["red", "green", "blue", "alpha"]

#@markdown ---
#@markdown ### PROMPT
#@markdown Fala da FUNCAO do inpaint (reconstruir o figurino), nunca da
#@markdown personagem. O design especifico vem da imagem e da referencia.
PROMPT_PRESET = "outfit_repair_v1"  #@param ["outfit_repair_v1"]
USAR_PROMPT_CUSTOM = False  #@param {type:"boolean"}
PROMPT_CUSTOM = "1girl, solo, full body, chibi, super deformed, clean lineart, anime coloring, simple cel shading, detailed outfit, detailed costume, preserve character design"  #@param {type:"string"}
#@markdown
USAR_NEGATIVE_CUSTOM = False  #@param {type:"boolean"}
NEGATIVE_CUSTOM = "bad quality, worst quality, worst detail, sketch, watermark, signature, logo, text, realistic, photorealistic, 3d, nude, naked, different outfit, changed design, extra limbs, deformed"  #@param {type:"string"}

#@markdown ---
#@markdown ### INPAINT STRENGTH
#@markdown EXPERIMENTAL. Nao ha motivo para herdar o 0.90 do img2img: aqui o
#@markdown objetivo e corrigir, nao retransformar. Sugestoes: 0.50 0.60 0.70
#@markdown 0.75 0.80 0.90. Sem sweep automatico nesta fase.
INPAINT_STRENGTH = 0.75  #@param {type:"slider", min:0.10, max:1.00, step:0.05}

#@markdown ---
#@markdown ### SAMPLING (exemplo oficial do model card: 28 steps, CFG 5.0)
STEPS = 28  #@param {type:"slider", min:10, max:60, step:1}
CFG_SCALE = 5.0  #@param {type:"slider", min:1.0, max:12.0, step:0.5}
SAMPLER = "euler_ancestral"  #@param {type:"string"}
SCHEDULER = "normal"  #@param {type:"string"}
SEED = 42  #@param {type:"integer"}

#@markdown ---
#@markdown ### V-PREDICTION
#@markdown O modelo deriva do WAI V14.0 **V-Prediction**, nao do nosso v17.0
#@markdown (eps). Com `eps` o output sai queimado. `zsnr` nao e declarado pelo
#@markdown autor — nem todo v-pred usa, e aplicar indevidamente tambem queima.
SAMPLING_TYPE = "v_prediction"  #@param ["v_prediction", "eps"]
ZSNR = False  #@param {type:"boolean"}

#@markdown ---
#@markdown ### SAIDA
OUTPUT_RESOLUTION = "match_source"  #@param ["match_source", "1024x1024"]
EXPERIMENT_LABEL = ""  #@param {type:"string"}

# ----------------------------------------------------------------------
import datetime, pathlib

MODEL_KEY = "waifu_inpaint_xl"
WORKFLOW = "experimental/waifu_inpaint_xl"
TESTE = "TESTE_1_INPAINT_PURO"

PROMPT_PRESETS = {
    "outfit_repair_v1": {
        "positive": (
            "1girl, solo, full body, chibi, super deformed, clean lineart, "
            "anime coloring, simple cel shading, detailed outfit, "
            "detailed costume, preserve character design"
        ),
        "negative": (
            "bad quality, worst quality, worst detail, sketch, watermark, "
            "signature, logo, text, realistic, photorealistic, 3d, nude, "
            "naked, different outfit, changed design, extra limbs, deformed"
        ),
    },
}

_preset = PROMPT_PRESETS[PROMPT_PRESET]

def _escolhe(usar, texto, padrao, lado):
    if not usar:
        return padrao, f"preset:{PROMPT_PRESET}"
    t = texto.strip()
    assert t, f"USAR_{lado}_CUSTOM marcado mas o campo esta vazio."
    if t == padrao.strip():
        return padrao, f"preset:{PROMPT_PRESET}"
    return t, "manual_override"

PROMPT, _src_pos = _escolhe(USAR_PROMPT_CUSTOM, PROMPT_CUSTOM,
                            _preset["positive"], "PROMPT")
NEGATIVE, _src_neg = _escolhe(USAR_NEGATIVE_CUSTOM, NEGATIVE_CUSTOM,
                              _preset["negative"], "NEGATIVE")
PROMPT_SOURCE = {"positive": _src_pos, "negative": _src_neg}
PROMPT_EDITADO = "manual_override" in PROMPT_SOURCE.values()

# O TESTE 2 (com referencia) precisa de IP-Adapter sobre um modelo de 9
# canais, o que e uma combinacao que ainda nao validamos. Prometer a opcao no
# dropdown e rodar inpaint puro seria mentir sobre o que foi executado.
assert REFERENCE_MODE == "NONE", (
    f"REFERENCE_MODE={REFERENCE_MODE} pertence ao TESTE 2 e ainda NAO esta "
    "implementado. O TESTE 1 (baseline, inpaint puro) precisa vir primeiro: "
    "sem ele nao da para saber o que o modelo faz sozinho. Use NONE.")

assert 0.0 < INPAINT_STRENGTH <= 1.0, "INPAINT_STRENGTH fora de (0, 1]"
assert MASK_DILATION >= 0 and MASK_FEATHER >= 0

CHAR_DIR = pathlib.Path("/content/ChibiCreate/characters") / CHARACTER_ID
LAB_ROOT = pathlib.Path("/content/ChibiCreate/experiments/inpaint_lab")

STAMP = datetime.datetime.now().strftime("%Y%m%d_%H%M%S")
_auto = f"inpaint_s{INPAINT_STRENGTH:.2f}_d{MASK_DILATION}_f{MASK_FEATHER}"
EXPERIMENT_NAME = EXPERIMENT_LABEL.strip() or _auto
EXPERIMENT_DIR_NAME = f"experiment_{STAMP}"

CONFIG = {
    "experiment": EXPERIMENT_NAME,
    "experiment_dir": EXPERIMENT_DIR_NAME,
    "test": TESTE,
    "created_utc": datetime.datetime.now(datetime.timezone.utc).isoformat(),
    "character_id": CHARACTER_ID,
    "model_key": MODEL_KEY,
    "pipeline": "sdxl_inpaint_9ch",
    "hypothesis": (
        "Inpaint localizado corrige o figurino sem destruir a personagem "
        "ja transformada em chibi."),
    "source_image": SOURCE_IMAGE,
    "source_image_role": "chibi_already_generated",
    "outfit_mask": OUTFIT_MASK,
    "mask_role": "semantic_region_that_may_be_redrawn",
    "reference_mode": REFERENCE_MODE,
    "references_available_not_used": {
        "outfit": OUTFIT_REFERENCE, "full_body": FULL_BODY_REFERENCE,
        "face": FACE_REFERENCE,
    },
    "prompt_preset": PROMPT_PRESET,
    "prompt": PROMPT,
    "negative_prompt": NEGATIVE,
    "prompt_source": PROMPT_SOURCE,
    "prompt_manually_edited": PROMPT_EDITADO,
    "prompt_type": "generic_outfit_repair",
    "character_specific_prompt": False,
    "mask": {"dilation": int(MASK_DILATION), "feather": int(MASK_FEATHER),
             "channel": MASK_CHANNEL},
    "sampling": {"seed": int(SEED), "steps": int(STEPS),
                 "cfg": float(CFG_SCALE), "sampler": SAMPLER,
                 "scheduler": SCHEDULER,
                 "sampling_type": SAMPLING_TYPE, "zsnr": bool(ZSNR)},
    "inpaint_strength": float(INPAINT_STRENGTH),
    "inpaint_strength_note": (
        "EXPERIMENTAL. Nao herdado do img2img: la o objetivo era transformar, "
        "aqui e corrigir."),
    "output_resolution": OUTPUT_RESOLUTION,
    "status": "BASELINE_EXPERIMENTAL",
}

print("=" * 66)
print("WAI INPAINT-XL LAB —", TESTE)
print("=" * 66)
print("experimento :", EXPERIMENT_NAME)
print("diretorio   :", EXPERIMENT_DIR_NAME)
print()
print("TRES ENTRADAS DISTINTAS (nao confundir):")
print("  SOURCE_IMAGE    :", SOURCE_IMAGE, "-> chibi ja gerado")
print("  OUTFIT_MASK     :", OUTFIT_MASK, "-> onde PODE redesenhar")
print("  OUTFIT_REFERENCE:", OUTFIT_REFERENCE, "-> design (NAO usado no TESTE 1)")
print()
print("MODELO   :", MODEL_KEY, "| UNet 9 canais |", SAMPLING_TYPE,
      "| zsnr", ZSNR)
print("MASCARA  : dilation", MASK_DILATION, "| feather", MASK_FEATHER,
      "| canal", MASK_CHANNEL)
print("SAMPLING : steps", STEPS, "| cfg", CFG_SCALE, "|", SAMPLER, "/",
      SCHEDULER, "| seed", SEED)
print("STRENGTH :", INPAINT_STRENGTH, "(experimental)")
print()
print("PROMPT   :", PROMPT[:90])
print("  origem :", PROMPT_SOURCE["positive"])
print()
print("Referencia:", REFERENCE_MODE, "-> TESTE 1 e inpaint PURO, sem IP-Adapter.")
print("Objetivo  : medir o que o modelo faz sozinho antes de somar variaveis.")


In [ ]:
#@title 1. Verificar o ambiente e o modelo { display-mode: "form" }
#@markdown Confere GPU, ComfyUI e — principalmente — se o checkpoint do
#@markdown Waifu-Inpaint-XL existe. O repo e GATED: o download exige aceite
#@markdown manual dos termos e um token HF. Nao ha como automatizar isso.
import subprocess, pathlib, json, hashlib

print("GPU:")
print(subprocess.run(["nvidia-smi", "--query-gpu=name,memory.total",
                      "--format=csv,noheader"],
                     capture_output=True, text=True).stdout.strip() or "NAO DETECTADA")

CKPT_DIR = pathlib.Path("/content/drive/MyDrive/ComfyUI_Data/models/checkpoints")
INPAINT_CKPT_NAME = "Waifu-Inpaint-XL.safetensors"
INPAINT_CKPT = CKPT_DIR / INPAINT_CKPT_NAME

print()
print("checkpoint esperado:", INPAINT_CKPT)
if not INPAINT_CKPT.exists():
    raise SystemExit(
        "BLOCKED — Waifu-Inpaint-XL.safetensors nao encontrado.\n"
        "\n"
        "O repositorio no Hugging Face e GATED: exige aceitar os termos e\n"
        "compartilhar contato antes de liberar os arquivos. Isso e um ato\n"
        "pessoal, feito na sua conta — o agente nao pode aceitar por voce,\n"
        "nem contornar o gate.\n"
        "\n"
        "Passos:\n"
        "  1. abra https://huggingface.co/ShinoharaHare/Waifu-Inpaint-XL\n"
        "  2. faca login e aceite as condicoes\n"
        "  3. baixe Waifu-Inpaint-XL.safetensors (6.94 GB)\n"
        f"  4. coloque em {CKPT_DIR}\n"
        "\n"
        "Confira tambem a licenca (CreativeML Open RAIL++-M) antes de\n"
        "qualquer uso comercial.")

_h = hashlib.sha256()
with open(INPAINT_CKPT, "rb") as f:
    for bloco in iter(lambda: f.read(1 << 22), b""):
        _h.update(bloco)
INPAINT_CKPT_SHA256 = _h.hexdigest()
print("tamanho :", INPAINT_CKPT.stat().st_size, "bytes")
print("sha256  :", INPAINT_CKPT_SHA256)
print()
print("[registrar no models.lock.yaml — o peso NUNCA vai para o Git]")


In [ ]:
#@title 2. Validar as entradas e a mascara { display-mode: "form" }
#@markdown A mascara e o fator mais critico do experimento. Aqui checamos
#@markdown que ela existe, casa com a source e cobre uma fracao plausivel da
#@markdown imagem — mascara ocupando quase tudo significa redesenhar a
#@markdown personagem inteira, que e exatamente o que esta fase evita.
import pathlib, hashlib, numpy as np
from PIL import Image

SRC = CHAR_DIR / "chibi" / SOURCE_IMAGE
MASK = CHAR_DIR / "masks" / OUTFIT_MASK

def _sha(p):
    h = hashlib.sha256()
    with open(p, "rb") as f:
        for b in iter(lambda: f.read(1 << 20), b""):
            h.update(b)
    return h.hexdigest()

falhas = []
for rotulo, caminho in (("SOURCE_IMAGE", SRC), ("OUTFIT_MASK", MASK)):
    if not caminho.exists():
        falhas.append(f"{rotulo} nao encontrado: {caminho}")

if falhas:
    raise SystemExit(
        "BLOCKED — " + "; ".join(falhas) + "\n\n"
        "A mascara e MANUAL nesta fase, de proposito: segmentacao automatica "
        "e outro problema, e uma mascara ruim invalidaria o experimento "
        "inteiro. Pinte de BRANCO a roupa e de PRETO o resto (rosto, cabelo, "
        "olhos, chifres, pele, membros fora da roupa).")

src_img = Image.open(SRC).convert("RGB")
mask_img = Image.open(MASK).convert("L")
SOURCE_SHA256, MASK_SHA256 = _sha(SRC), _sha(MASK)

print("source:", SRC.name, src_img.size, "sha", SOURCE_SHA256[:16])
print("mask  :", MASK.name, mask_img.size, "sha", MASK_SHA256[:16])

if src_img.size != mask_img.size:
    raise SystemExit(
        f"BLOCKED — mascara {mask_img.size} != source {src_img.size}. "
        "Redimensionar automaticamente deslocaria a regiao protegida.")

m = np.asarray(mask_img) > 127
MASK_AREA_PCT = round(100.0 * m.sum() / m.size, 2)
print("area mascarada:", MASK_AREA_PCT, "%")

if not m.any():
    raise SystemExit("BLOCKED — mascara totalmente preta: nada a inpaintar.")
if MASK_AREA_PCT > 60:
    raise SystemExit(
        f"BLOCKED — mascara cobre {MASK_AREA_PCT}% da imagem. Isso deixa de "
        "ser correcao localizada e vira retransformacao da personagem, que e "
        "justamente o que esta fase quer evitar.")
if MASK_AREA_PCT < 1:
    raise SystemExit(
        f"BLOCKED — mascara cobre so {MASK_AREA_PCT}%. Provavel erro de canal "
        "ou de exportacao.")

print()
print("[HUMAN REVIEW REQUIRED] confira visualmente que a mascara NAO invade")
print("rosto, cabelo, olhos, chifres, pele ou membros fora da roupa.")
display(Image.blend(src_img, Image.merge("RGB", (mask_img, mask_img, mask_img)), 0.5)
        .resize((384, 384)))


In [ ]:
#@title 3. Validar nodes e montar o grafo { display-mode: "form" }
#@markdown Valida contra o `/object_info` REAL. O ponto critico:
#@markdown `InpaintModelConditioning` (nao `VAEEncodeForInpaint`), porque so
#@markdown ele permite strength < 1.0 preservando o conteudo sob a mascara.
import json, urllib.request, pathlib, hashlib

OBJECT_INFO = json.load(urllib.request.urlopen(
    "http://127.0.0.1:8188/object_info", timeout=120))

falhas = []
def checa(cond, ok, erro):
    print(("  OK    " if cond else "  FALHA ") + (ok if cond else erro))
    if not cond:
        falhas.append(erro)

NECESSARIOS = ["CheckpointLoaderSimple", "ModelSamplingDiscrete",
               "CLIPTextEncode", "LoadImage", "LoadImageMask", "GrowMask",
               "FeatherMask", "InpaintModelConditioning", "KSampler",
               "VAEDecode", "SaveImage"]
print("NODES")
for n in NECESSARIOS:
    checa(n in OBJECT_INFO, f"{n} disponivel", f"{n} AUSENTE")

def _opcoes(node, campo):
    for grupo in ("required", "optional"):
        spec = OBJECT_INFO.get(node, {}).get("input", {}).get(grupo, {})
        if campo in spec and isinstance(spec[campo][0], list):
            return spec[campo][0]
    return None

print()
print("PARAMETROS (contra o servidor, nao contra suposicao)")
for node, campo, valor in (("KSampler", "sampler_name", SAMPLER),
                           ("KSampler", "scheduler", SCHEDULER),
                           ("ModelSamplingDiscrete", "sampling", SAMPLING_TYPE),
                           ("LoadImageMask", "channel", MASK_CHANNEL)):
    opts = _opcoes(node, campo)
    if opts is None:
        checa(False, "", f"{node}.{campo} nao encontrado no /object_info")
    else:
        checa(valor in opts, f"{node}.{campo}={valor}",
              f"{node}.{campo}={valor} inexistente. Disponiveis: {opts}")

# zsnr so existe em versoes recentes; se faltar, dizer em vez de ignorar.
if "ModelSamplingDiscrete" in OBJECT_INFO:
    tem_zsnr = _opcoes("ModelSamplingDiscrete", "zsnr") is not None or (
        "zsnr" in OBJECT_INFO["ModelSamplingDiscrete"]["input"].get("required", {}))
    if not tem_zsnr and ZSNR:
        checa(False, "", "ZSNR=True mas este ComfyUI nao expoe o parametro zsnr")

WF_PATH = pathlib.Path(f"/content/ChibiCreate/workflows/{WORKFLOW}/v0.json")
WF_RAW = json.loads(WF_PATH.read_text())
WORKFLOW_SHA256 = hashlib.sha256(WF_PATH.read_bytes()).hexdigest()
WF = {k: v for k, v in WF_RAW.items() if not k.startswith("_")}
cls = {k: v["class_type"] for k, v in WF.items()}

print()
print("ESTRUTURA DO GRAFO")
ks = [k for k, c in cls.items() if c == "KSampler"][0]
imc = [k for k, c in cls.items() if c == "InpaintModelConditioning"]
checa(len(imc) == 1, "InpaintModelConditioning presente",
      "grafo nao usa InpaintModelConditioning")
checa("VAEEncodeForInpaint" not in cls.values(),
      "nao usa VAEEncodeForInpaint",
      "VAEEncodeForInpaint exige denoise 1.0 e destruiria o design existente")
checa(WF[ks]["inputs"]["latent_image"][0] in imc,
      "latente vem do InpaintModelConditioning",
      "latente nao vem do InpaintModelConditioning")
checa(cls[WF[ks]["inputs"]["model"][0]] == "ModelSamplingDiscrete",
      "modelo passa por ModelSamplingDiscrete (v-pred)",
      "modelo nao declara v_prediction: output sairia queimado")
_mask_src = WF[imc[0]]["inputs"]["mask"][0]
checa(cls[_mask_src] == "FeatherMask", "mascara passa por feather",
      "mascara nao passa por feather")
_grow = WF[cls[_mask_src] and WF[_mask_src]["inputs"]["mask"][0]]
checa(_grow["class_type"] == "GrowMask", "mascara passa por dilatacao",
      "mascara nao passa por GrowMask")
checa(cls[WF[imc[0]]["inputs"]["pixels"][0]] == "LoadImage",
      "pixels vem da SOURCE_IMAGE", "pixels nao vem de LoadImage")
_ausentes = [c for c in set(cls.values()) if c not in OBJECT_INFO]
checa(not _ausentes, "todas as classes existem no servidor",
      f"classes ausentes: {_ausentes}")

print()
if falhas:
    raise SystemExit(f"BLOCKED — {len(falhas)} falha(s): {falhas}")
print("Grafo valido. Nenhuma substituicao silenciosa de node.")


In [ ]:
#@title 4. Mostrar o grafo resolvido (antes de executar) { display-mode: "form" }
import json, copy

SUBS = {
    "%%INPAINT_CKPT%%": INPAINT_CKPT_NAME,
    "%%SAMPLING_TYPE%%": SAMPLING_TYPE,
    "%%ZSNR%%": bool(ZSNR),
    "%%PROMPT%%": PROMPT,
    "%%NEGATIVE_PROMPT%%": NEGATIVE,
    "%%SOURCE_IMAGE%%": SRC.name,
    "%%OUTFIT_MASK%%": MASK.name,
    "%%MASK_CHANNEL%%": MASK_CHANNEL,
    "%%MASK_DILATION%%": int(MASK_DILATION),
    "%%MASK_FEATHER%%": int(MASK_FEATHER),
    "%%SEED%%": int(SEED),
    "%%STEPS%%": int(STEPS),
    "%%CFG%%": float(CFG_SCALE),
    "%%SAMPLER%%": SAMPLER,
    "%%SCHEDULER%%": SCHEDULER,
    "%%INPAINT_STRENGTH%%": float(INPAINT_STRENGTH),
    "%%OUTPUT_PREFIX%%": f"inpaint_{EXPERIMENT_DIR_NAME}",
}

def resolver(wf):
    out = copy.deepcopy(wf)
    for node in out.values():
        for campo, valor in node["inputs"].items():
            if isinstance(valor, str) and valor in SUBS:
                node["inputs"][campo] = SUBS[valor]
    return out

GRAFO = resolver(WF)
restantes = [f"{k}.{c}={v}" for k, n in GRAFO.items()
             for c, v in n["inputs"].items()
             if isinstance(v, str) and v.startswith("%%")]
assert not restantes, f"placeholders nao resolvidos: {restantes}"

print(json.dumps(GRAFO, indent=2, ensure_ascii=False))
print()
print("CONFIRA ACIMA antes de executar a proxima celula.")


In [ ]:
#@title 5. Executar o BASELINE (uma unica configuracao) { display-mode: "form" }
#@markdown Executa **um** experimento. Sem sweep: a prioridade e validar a
#@markdown arquitetura, nao varrer parametros.
import json, urllib.request, time, pathlib, shutil, hashlib, subprocess

EXP_DIR = LAB_ROOT / EXPERIMENT_DIR_NAME
if EXP_DIR.exists():
    raise SystemExit(f"BLOCKED — {EXP_DIR} ja existe. Nao sobrescrevemos.")
for sub in ("source", "mask", "reference", "output", "logs"):
    (EXP_DIR / sub).mkdir(parents=True)

shutil.copy2(SRC, EXP_DIR / "source" / SRC.name)
shutil.copy2(MASK, EXP_DIR / "mask" / MASK.name)
(EXP_DIR / "config.json").write_text(json.dumps(CONFIG, indent=2, ensure_ascii=False))
(EXP_DIR / "workflow.resolved.json").write_text(json.dumps(GRAFO, indent=2, ensure_ascii=False))

# O ComfyUI le as imagens do proprio input/
COMFY_IN = pathlib.Path("/content/ComfyUI/input")
shutil.copy2(SRC, COMFY_IN / SRC.name)
shutil.copy2(MASK, COMFY_IN / MASK.name)

t0 = time.time()
req = urllib.request.Request(
    "http://127.0.0.1:8188/prompt",
    data=json.dumps({"prompt": GRAFO}).encode(),
    headers={"Content-Type": "application/json"})
pid = json.load(urllib.request.urlopen(req))["prompt_id"]
print("prompt_id:", pid)

hist = {}
while time.time() - t0 < 1800:
    hist = json.load(urllib.request.urlopen(
        f"http://127.0.0.1:8188/history/{pid}", timeout=30))
    if pid in hist:
        break
    time.sleep(2)
else:
    raise SystemExit("BLOCKED — timeout de 1800s")

ELAPSED = round(time.time() - t0, 1)
st = hist[pid].get("status", {})
if st.get("status_str") == "error":
    (EXP_DIR / "logs" / "error.json").write_text(json.dumps(hist[pid], indent=2))
    raise SystemExit(f"BLOCKED — execucao falhou. Log em {EXP_DIR}/logs/error.json")

imgs = [i for o in hist[pid]["outputs"].values() for i in o.get("images", [])]
assert imgs, "nenhuma imagem retornada"
src_out = pathlib.Path("/content/ComfyUI/output") / imgs[0]["filename"]
OUT_PATH = EXP_DIR / "output" / "output.png"
shutil.copy2(src_out, OUT_PATH)

def _sha(p):
    h = hashlib.sha256()
    with open(p, "rb") as f:
        for b in iter(lambda: f.read(1 << 20), b""):
            h.update(b)
    return h.hexdigest()

from PIL import Image
import numpy as np
_px = hashlib.sha256(
    np.asarray(Image.open(OUT_PATH).convert("RGB")).tobytes()).hexdigest()

def _commit(d):
    try:
        return subprocess.run(["git", "-C", d, "rev-parse", "HEAD"],
                              capture_output=True, text=True).stdout.strip()
    except Exception:
        return "unknown"

RECIPE = {
    **CONFIG,
    "elapsed_seconds": ELAPSED,
    "model": {
        "key": MODEL_KEY,
        "file": INPAINT_CKPT_NAME,
        "sha256": INPAINT_CKPT_SHA256,
        "repo": "ShinoharaHare/Waifu-Inpaint-XL",
        "lineage": ["KBlueLeaf/kohaku-xl-beta5",
                    "OnomaAIResearch/Illustrious-xl-early-release-v0",
                    "ShinoharaHare/WAI-NSFW-illustrious-SDXL-V14.0-V-Prediction",
                    "ShinoharaHare/Waifu-Inpaint-XL"],
        "unet_in_channels": 9,
        "license": "CreativeML Open RAIL++-M",
        "gated": True,
    },
    "inputs": {
        "source_sha256": SOURCE_SHA256,
        "mask_sha256": MASK_SHA256,
        "mask_area_percentage": MASK_AREA_PCT,
        "reference_used": None,
    },
    "workflow_sha256": WORKFLOW_SHA256,
    "artifact_sha256": _sha(OUT_PATH),
    "output_pixel_sha256": _px,
    "comfyui_commit": _commit("/content/ComfyUI"),
    "custom_nodes": [],
    "custom_nodes_note": "TESTE 1 usa apenas nodes de fabrica do ComfyUI.",
    "loader_node": "CheckpointLoaderSimple",
    "conditioning_node": "InpaintModelConditioning",
    "determinism_note": (
        "Mesma seed e mesmo grafo tendem a reproduzir, mas GPU e versao de "
        "biblioteca podem alterar bits. Nao prometemos determinismo absoluto."),
}
(EXP_DIR / "recipe.json").write_text(json.dumps(RECIPE, indent=2, ensure_ascii=False))
print("OK em", ELAPSED, "s ->", OUT_PATH)


In [ ]:
#@title 6. Medir a LOCALIDADE do inpaint { display-mode: "form" }
#@markdown Responde a pergunta tecnica da fase: **o inpaint mexeu so onde a
#@markdown mascara permitia?** Nao julga beleza.
import sys, json
sys.path.insert(0, "/content/ChibiCreate/scripts")
from chibi.inpaint_check import comparar, diferenca_visivel
from PIL import Image
import matplotlib.pyplot as plt

METRICAS = comparar(SRC, OUT_PATH, MASK)
print(json.dumps(METRICAS, indent=2, ensure_ascii=False))

src_i = Image.open(SRC).convert("RGB")
out_i = Image.open(OUT_PATH).convert("RGB")
mask_i = Image.open(MASK).convert("L")
dif = diferenca_visivel(SRC, OUT_PATH)

fig, ax = plt.subplots(1, 4, figsize=(18, 5))
for a, img, t in ((ax[0], src_i, "SOURCE (chibi)"),
                  (ax[1], mask_i, "MASK"),
                  (ax[2], out_i, "OUTPUT"),
                  (ax[3], dif, "DIFERENCA (x8)")):
    a.imshow(img, cmap="gray" if img.mode == "L" else None)
    a.set_title(t, fontsize=11)
    a.axis("off")
fig.suptitle(
    f"preservado fora da mascara: "
    f"{METRICAS['outside_mask_preserved_percentage']}%  |  "
    f"area mascarada: {METRICAS['mask_area_percentage']}%", fontsize=12)
fig.tight_layout()
CMP = EXP_DIR / "comparison.png"
fig.savefig(CMP, dpi=110, bbox_inches="tight")
plt.show()

(EXP_DIR / "comparison.json").write_text(
    json.dumps({"metrics": METRICAS, "recipe": RECIPE}, indent=2, ensure_ascii=False))
(EXP_DIR / "hashes.json").write_text(json.dumps({
    "checkpoint_sha256": INPAINT_CKPT_SHA256,
    "source_sha256": SOURCE_SHA256,
    "mask_sha256": MASK_SHA256,
    "workflow_sha256": WORKFLOW_SHA256,
    "artifact_sha256": RECIPE["artifact_sha256"],
    "output_pixel_sha256": RECIPE["output_pixel_sha256"],
}, indent=2))

print()
print("LEITURA TECNICA (nao artistica):")
p = METRICAS["outside_mask_preserved_percentage"]
print(f"  {p}% dos pixels fora da mascara continuam inalterados.")
if p < 95:
    print("  ATENCAO: o inpaint alterou area fora da mascara de forma")
    print("  significativa. Ha relato de terceiros de que este modelo desloca")
    print("  levemente a cor da imagem inteira. Registrar como achado — nao")
    print("  compensar com pos-processamento.")
print()
print("[HUMAN REVIEW REQUIRED] o design do figurino ficou aceitavel?")
print("  A metrica acima nao responde isso. Avaliacao estetica e humana.")


In [ ]:
#@title 7. Empacotar o experimento { display-mode: "form" }
import shutil, json
ZIP_BASE = f"/content/inpaint_lab_{EXPERIMENT_DIR_NAME}"
ZIP = shutil.make_archive(ZIP_BASE, "zip", root_dir=str(EXP_DIR))
print("ZIP:", ZIP)
print()
print("conteudo:")
for p in sorted(EXP_DIR.rglob("*")):
    if p.is_file():
        print("  ", p.relative_to(EXP_DIR), f"({p.stat().st_size} bytes)")
try:
    from google.colab import files
    files.download(ZIP)
except Exception as e:
    print("download manual:", ZIP, "|", e)
